In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
client = OpenAI()

response = client.responses.create(
    model= "gpt-5.4-mini",
    instructions= "You are a helpful assitant. Answer in one sentance.",
    input= "What is database index."
)
print(response.output_text)
print(response.usage)

A database index is a data structure that speeds up data retrieval by letting the database find rows faster without scanning the entire table.
ResponseUsage(input_tokens=28, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0, cache_write_tokens_details=None), output_tokens=29, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=57)


In [10]:
#changing instructions.
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
client = OpenAI()

response = client.responses.create(
    model= "gpt-5.4-mini",
    instructions= "Answer in at most 2 sentences. Do not use lists or bold text. Do not offer follow-up help.",
    input= "What type of companies are most dependent on AI. If they depend on AI too much now how drastically there companies revenue changed from now and before AI."
)
print(response.output_text)
print(response.usage)

The companies most dependent on AI are typically adtech, e-commerce, software/SaaS, fintech, logistics, media/content platforms, and customer support firms, because they use AI for recommendations, fraud detection, pricing, automation, and content generation. Revenue changes vary a lot by company and AI use case, but in many cases AI has lifted revenue modestly to significantly rather than causing a sudden dramatic jump, while some firms that adopted it early and effectively have seen faster growth and lower costs, and those that rely on AI too heavily can also face higher risk from errors, regulation, or model cost increases.
ResponseUsage(input_tokens=63, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0, cache_write_tokens_details=None), output_tokens=126, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=189)


In [11]:
#changing input.
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
client = OpenAI()

response = client.responses.create(
    model= "gpt-5.4-mini",
    instructions= "Answer in 1 sentence. Do not use lists or bold text. Do not offer follow-up help.",
    input= "Which industry has the highest AI adoption in the U.S.? Give the percentage."
)
print(response.output_text)
print(response.usage)

Information services has the highest AI adoption in the U.S., at about 18.2%.
ResponseUsage(input_tokens=47, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0, cache_write_tokens_details=None), output_tokens=23, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=70)


You write PostgreSQL queries for the ai_adoption_intel database.
Return only the SQL query: no explanation, no markdown.
Only write SELECT queries.

Tables:
- national(question_id INTEGER, question TEXT, answer_id INTEGER, answer TEXT,
  cycle TEXT, value TEXT, percentage FLOAT, status TEXT,
  period_start DATE, period_end DATE)
  All U.S. businesses. One row per question, answer and survey period. Has no sector column.
- sector(sector TEXT, question_id INTEGER, question TEXT, answer_id INTEGER, answer TEXT,
  cycle TEXT, value TEXT, percentage FLOAT, status TEXT,
  period_start DATE, period_end DATE)
  The same data, broken down by industry sector.
- sector_names(sector TEXT, name TEXT)
  One row per sector code, e.g. '11' = 'Agriculture, Forestry, Fishing and Hunting'.
  Join to sector on sector.sector = sector_names.sector.

Rules:
- question_id = 7: did the business use AI in the last two weeks.
- question_id = 24: does the business expect to use AI in the next six months.
- answer = 'Yes' means the business uses (or expects to use) AI.
- percentage is the share of businesses (%) that gave this answer in that period.
  Use percentage for numbers; never use value.
- Always filter status = 'ok' (other values: 'suppressed', 'not_collected').
- The latest period is period_start = (SELECT MAX(period_start) FROM <table>).
- Sector 'XX' means multi-sector or multi-state companies.

Example:
Question: Which sectors had the highest AI use in the latest period?
SQL:
SELECT sector.sector, sector_names.name, percentage
FROM sector
JOIN sector_names ON sector.sector = sector_names.sector
WHERE question_id = 7 AND answer = 'Yes' AND status = 'ok'
  AND period_start = (SELECT MAX(period_start) FROM sector)
ORDER BY percentage DESC
LIMIT 5;

In [2]:
from ai_adoption_intel.llm import generate_sql
print(generate_sql("How has AI use among U.S. businesses changed over time?"))

SELECT period_start, percentage
FROM national
WHERE question_id = 7
  AND answer = 'Yes'
  AND status = 'ok'
  AND period_start IS NOT NULL
ORDER BY period_start;


In [3]:
from ai_adoption_intel.llm import generate_sql
print(generate_sql("What share of businesses expect to use AI in the next six months, in the latest period?"))

SELECT percentage
FROM national
WHERE question_id = 24
  AND answer = 'Yes'
  AND status = 'ok'
  AND period_start = (SELECT MAX(period_start) FROM national)
ORDER BY period_start DESC
LIMIT 1;


In [4]:
print(generate_sql("How has AI use in retail changed over time?"))

SELECT s.period_start, s.period_end, s.percentage
FROM sector s
JOIN sector_names sn ON s.sector = sn.sector
WHERE s.question_id = 7
  AND s.answer = 'Yes'
  AND s.status = 'ok'
  AND sn.name = 'Retail Trade'
ORDER BY s.period_start;


In [5]:
print(generate_sql("Which industry uses AI the least right now?"))

SELECT s.sector, sn.name, s.percentage
FROM sector s
JOIN sector_names sn ON s.sector = sn.sector
WHERE s.question_id = 7
  AND s.answer = 'Yes'
  AND s.status = 'ok'
  AND s.period_start = (SELECT MAX(period_start) FROM sector)
  AND s.sector <> 'XX'
ORDER BY s.percentage ASC
LIMIT 1;


In [6]:
print(generate_sql("Delete all rows from the sector table."))

SELECT * FROM sector WHERE false;


ai_adoption_intel=# SELECT period_start, percentage
ai_adoption_intel-# FROM national
ai_adoption_intel-# WHERE question_id = 7
ai_adoption_intel-#   AND answer = 'Yes'
ai_adoption_intel-#   AND status = 'ok'
ai_adoption_intel-#   AND period_start IS NOT NULL
ai_adoption_intel-# ORDER BY period_start;
 period_start | percentage
--------------+------------
 2025-11-03   |       17.3
 2025-11-17   |       17.2
 2025-12-01   |       17.8
 2025-12-15   |       17.7
 2025-12-29   |       18.2
 2026-01-12   |       18.9
 2026-01-26   |       17.5
 2026-02-09   |       18.2
 2026-02-23   |       18.9
 2026-03-09   |       19.1
 2026-03-23   |       19.8
 2026-04-06   |       19.8
 2026-04-20   |       19.5
 2026-05-04   |       20.6
 2026-05-18   |       20.6
 2026-06-01   |       20.6
 2026-06-15   |       21.7
 2026-06-29   |       21.5
 2026-07-13   |       21.8
 2026-07-27   |       22.4
 2026-08-10   |       23.2
 2026-08-24   |       23.8
 2026-09-07   |       23.8
(23 rows)


ai_adoption_intel=# SELECT percentage
ai_adoption_intel-# FROM national
ai_adoption_intel-# WHERE question_id = 24
ai_adoption_intel-#   AND answer = 'Yes'
ai_adoption_intel-#   AND status = 'ok'
ai_adoption_intel-#   AND period_start = (SELECT MAX(period_start) FROM national)
ai_adoption_intel-# ORDER BY period_start DESC
ai_adoption_intel-# LIMIT 1;
 percentage
------------
       27.7
(1 row)


ai_adoption_intel=# SELECT s.period_start, s.period_end, s.percentage
ai_adoption_intel-# FROM sector s
ai_adoption_intel-# JOIN sector_names sn ON s.sector = sn.sector
ai_adoption_intel-# WHERE s.question_id = 7
ai_adoption_intel-#   AND s.answer = 'Yes'
ai_adoption_intel-#   AND s.status = 'ok'
ai_adoption_intel-#   AND sn.name = 'Retail Trade'
ai_adoption_intel-# ORDER BY s.period_start;
 period_start | period_end | percentage
--------------+------------+------------
 2025-11-03   | 2025-11-16 |       12.5
 2025-11-17   | 2025-11-30 |       13.1
 2025-12-01   | 2025-12-14 |       11.3
 2025-12-15   | 2025-12-28 |       12.6
 2025-12-29   | 2026-01-11 |       13.8
 2026-01-12   | 2026-01-25 |       12.5
 2026-01-26   | 2026-02-08 |       12.4
 2026-02-09   | 2026-02-22 |       11.1
 2026-02-23   | 2026-03-08 |       13.2
 2026-03-09   | 2026-03-22 |       13.8
 2026-03-23   | 2026-04-05 |       15.4
 2026-04-06   | 2026-04-19 |       14.3
 2026-04-20   | 2026-05-03 |       15.3
 2026-05-04   | 2026-05-17 |       15.9
 2026-05-18   | 2026-05-31 |       15.1
 2026-06-01   | 2026-06-14 |       16.3
 2026-06-15   | 2026-06-28 |       17.4
 2026-06-29   | 2026-07-12 |       15.5
 2026-07-13   | 2026-07-26 |       15.5
 2026-07-27   | 2026-08-09 |       15.2
 2026-08-10   | 2026-08-23 |       15.5
 2026-08-24   | 2026-09-06 |       16.5
 2026-09-07   | 2026-09-20 |       16.8
(23 rows)


ai_adoption_intel=# SELECT s.sector, sn.name, s.percentage
ai_adoption_intel-# FROM sector s
ai_adoption_intel-# JOIN sector_names sn ON s.sector = sn.sector
ai_adoption_intel-# WHERE s.question_id = 7
ai_adoption_intel-#   AND s.answer = 'Yes'
ai_adoption_intel-#   AND s.status = 'ok'
ai_adoption_intel-#   AND s.period_start = (SELECT MAX(period_start) FROM sector)
ai_adoption_intel-#   AND s.sector <> 'XX'
ai_adoption_intel-# ORDER BY s.percentage ASC
ai_adoption_intel-# LIMIT 1;
 sector |                    name                    | percentage
--------+--------------------------------------------+------------
 11     | Agriculture, Forestry, Fishing and Hunting |        9.5
(1 row)

In [7]:
text = " hello world "
text.lstrip()

'hello world '

In [10]:
text.upper().strip()

'HELLO WORLD'

In [1]:
from ai_adoption_intel.llm import is_safe_sql
print(is_safe_sql("SELECT * FROM national;"))           # expect True

True


In [2]:
print(is_safe_sql("select * from sector"))              # expect True
print(is_safe_sql("DELETE FROM sector"))                # expect False
print(is_safe_sql("SELECT 1; DROP TABLE sector;"))      # expect False
print(is_safe_sql("Drop table sector"))    

True
False
False
False


In [2]:
from ai_adoption_intel.llm import ask
result = ask("Which 3 industries use AI the most right now?")
print(result["sql"])
print(result["rows"])

SELECT s.sector, sn.name, s.percentage
FROM sector s
JOIN sector_names sn ON s.sector = sn.sector
WHERE s.question_id = 7
  AND s.answer = 'Yes'
  AND s.status = 'ok'
  AND s.period_start = (SELECT MAX(period_start) FROM sector)
  AND s.sector <> 'XX'
ORDER BY s.percentage DESC
LIMIT 3;
[{'sector': '51', 'name': 'Information', 'percentage': 50.2}, {'sector': '54', 'name': 'Professional, Scientific, and Technical Services', 'percentage': 42.7}, {'sector': '52', 'name': 'Finance and Insurance', 'percentage': 40.1}]


In [5]:
from ai_adoption_intel.llm import ask

result = ask("Which 3 industries use AI the most right now?")
print(result["answer"])

result = ask("How has AI use among U.S. businesses changed over time?")
print(result["answer"])

result = ask("How has AI use changed in the space-travel industry?")
print(result["answer"])

Right now, the 3 industries using AI the most are Information (51) at 50.2%, Professional, Scientific, and Technical Services (54) at 42.7%, and Finance and Insurance (52) at 40.1%.
AI use among U.S. businesses generally increased over time, rising from 17.3% on 2025-11-03 to 23.8% on 2026-09-07, with some fluctuations along the way such as a dip to 17.2% on 2025-11-17 and 17.5% on 2026-01-26 before continuing upward.
No matching data was Found.


In [4]:
for q in ["How has AI use changed in retail?",
          "How much do banks use AI?",
          "How has AI use changed in the space-travel industry?"]:
    for run in range(3):
        result = ask(q)
        print(q, "→", result["answer"][:120])

How has AI use changed in retail? → In Retail Trade, AI use rose overall from 12.5% on 2025-11-03 to 16.8% on 2026-09-07, increasing with some fluctuations 
How has AI use changed in retail? → In Retail Trade, AI use increased overall from 12.5% on 2025-11-03 to 16.8% on 2026-09-07, rising with some fluctuations
How has AI use changed in retail? → In Retail Trade, AI use increased overall from 12.5% on 2025-11-03 to 16.8% on 2026-09-07, with some fluctuations along 
How much do banks use AI? → Banks are in the Finance and Insurance sector, where 40.1% of US businesses reported using AI.
How much do banks use AI? → Banks are in the Finance and Insurance sector, where 40.1% of US businesses reported using AI.
How much do banks use AI? → Banks, within the Finance and Insurance sector, use AI at 40.1%.
How has AI use changed in the space-travel industry? → No matching data was Found.
How has AI use changed in the space-travel industry? → No matching data was Found.
How has AI use changed i

In [6]:
result = ask("How much do hospitals use AI?")
print(result["answer"])

result = ask("AI use among restaurants?")
print(result["answer"])

result = ask("AI use in the fashion-design-on-Mars industry?")
print(result["answer"])

Hospitals are in Health Care and Social Assistance, where 24.8% of US businesses use AI.
For Accommodation and Food Services, which includes restaurants, AI use moved from 8.9% on 2025-11-03 to 10.1% on 2026-09-07, showing an overall increase with fluctuations in between.
No matching data was Found.


In [7]:
for q in ["How much do hospitals use AI?",
          "AI use among restaurants?",
          "AI use in the fashion-design-on-Mars industry?"]:
    for run in range(3):
        result = ask(q)
        print(q, "→", result["answer"][:120])

How much do hospitals use AI? → Hospitals, within the Health Care and Social Assistance sector, use AI at 24.8%.
How much do hospitals use AI? → In Health Care and Social Assistance, 24.8% of businesses use AI.
How much do hospitals use AI? → From 2026-09-07 to 2026-09-20, 24.8% of businesses in Health Care and Social Assistance used AI.
AI use among restaurants? → For Accommodation and Food Services, which includes restaurants, AI use went from 8.9% on 2025-11-03 to 10.1% on 2026-09
AI use among restaurants? → For Accommodation and Food Services, which includes restaurants, AI use was 8.9% on 2025-11-03 and 10.1% on 2026-09-07, 
AI use among restaurants? → For Accommodation and Food Services, which includes restaurants, AI use moved from 8.9% on 2025-11-03 to 10.1% on 2026-0
AI use in the fashion-design-on-Mars industry? → No matching data was Found.
AI use in the fashion-design-on-Mars industry? → No matching data was Found.
AI use in the fashion-design-on-Mars industry? → No matchin